In [ ]:
from datasets import load_dataset
import pandas as pd
from google.colab import drive
import os

# 挂载 Google Drive
drive.mount('/content/drive')

# 加载完整 BELLE 数据集（2M 中文版）
print("正在加载 BelleGroup/train_2M_CN 数据集...")
dataset = load_dataset("BelleGroup/train_2M_CN", split="train")
total_size = len(dataset)
print(f"数据集总大小: {total_size:,} 条")


Mounted at /content/drive
正在加载 BelleGroup/train_2M_CN 数据集...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train_2M_CN.json:   0%|          | 0.00/1.94G [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2000000 [00:00<?, ? examples/s]

数据集总大小: 2,000,000 条


In [ ]:

# 创建保存目录
save_dir = '/content/drive/MyDrive/belle_dataset'
os.makedirs(save_dir, exist_ok=True)

# === 2. GRPO 偏好生成集（用于生成候选回答）===
# 取倒数第 105500 到倒数第 501 条（共 10000 条）
grpo_start = total_size - 10500
grpo_end = total_size - 500
grpo_indices = range(grpo_start, grpo_end)

grpo_dataset = dataset.select(grpo_indices)
df_grpo = pd.DataFrame(grpo_dataset)
grpo_path = os.path.join(save_dir, 'belle_grpo_gen_10000.json')
df_grpo.to_json(grpo_path, orient='records', lines=False, force_ascii=False)
print(f"✅ GRPO 候选生成集已保存 ({len(df_grpo)} 条): {grpo_path}")

# === 3. 评估集（用于自动评估）===
# 取最后 500 条
eval_indices = range(total_size - 500, total_size)
eval_dataset = dataset.select(eval_indices)
df_eval = pd.DataFrame(eval_dataset)
eval_path = os.path.join(save_dir, 'belle_eval_500.json')
df_eval.to_json(eval_path, orient='records', lines=False, force_ascii=False)
print(f"✅ 评估集已保存 ({len(df_eval)} 条): {eval_path}")

# === 验证无重叠 ===
print("\n📊 数据划分验证:")
print(f"- 总数据量: {total_size}")
print(f"- GRPO 生成集: [{grpo_start}, {grpo_end})")
print(f"- 评估集: [{total_size - 500}, {total_size})")
print(f"- 与训练集 [0, 5000) 无重叠: {'是' if grpo_start >= 5000 else '否'}")

✅ GRPO 候选生成集已保存 (10000 条): /content/drive/MyDrive/belle_splits/belle_grpo_gen_10000.json
✅ 评估集已保存 (500 条): /content/drive/MyDrive/belle_splits/belle_eval_500.json

📊 数据划分验证:
- 总数据量: 2000000
- GRPO 生成集: [1989500, 1999500)
- 评估集: [1999500, 2000000)
- 与训练集 [0, 5000) 无重叠: 是


In [ ]:
# 下载 BELLE 数据集并保存到 Google Drive
from datasets import load_dataset
import pandas as pd
from google.colab import drive
import os

# 挂载 Google Drive
drive.mount('/content/drive')

# 选择 BELLE 数据集（较小版本便于查看）
dataset_name = "LLMs/BELLE-0.5M"  # 52万条数据，较易处理

print(f"正在加载 {dataset_name} 数据集...")
dataset = load_dataset("BelleGroup/train_2M_CN", split="train")

print("数据集结构:")
print(dataset)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
正在加载 LLMs/BELLE-0.5M 数据集...


README.md: 0.00B [00:00, ?B/s]

train_2M_CN.json:   0%|          | 0.00/1.94G [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2000000 [00:00<?, ? examples/s]

数据集结构:
Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 2000000
})


ValueError: Column 'train' doesn't exist.

In [ ]:

# 查看前几条数据
sample_data = dataset.select(range(5))
print("\n前5条数据样例:")
for i, example in enumerate(sample_data):
    print(f"\n--- 样例 {i+1} ---")
    print(f"instruction: {example.get('instruction', 'N/A')}")
    print(f"input: {example.get('input', 'N/A')}")
    print(f"output: {example.get('output', 'N/A')}")



前5条数据样例:

--- 样例 1 ---
instruction: 对以下代码进行调试。
def add(a,b):
  return a + b
  print(add(2,3))
input: 
output: 这段代码的问题是在函数内部， `print` 语句出现在 `return` 语句之后，也就是说函数在 `return` 语句处结束，因此 `print` 语句永远不会被执行。
要解决这个问题，只需在 `return` 语句之前移到 `print` 语句。修改后的代码如下所示：
```
def add(a, b):
    print(a + b)
    return a + b
add(2, 3)
```
这样就可以在调用 `add` 函数时打印出结果，并在函数内部返回 `a + b` 值，使其在需要时可以接收和使用。同时，注意在调用 `add` 函数时，不需要再使用 `print` 方法了。

--- 样例 2 ---
instruction: 为一个新的智能手机应用程序起个优秀的名称并写一句简短的介绍。

input: 
output: 为一个新的智能手机应用程序起一个优秀的名称可以考虑以下几点：
1. 名称应该简单易记，不宜太长；
2. 名称应该与应用程序的主要功能或特点相关；
3. 名称应该引起人们的兴趣和好奇心。
举个例子，如果我们的应用程序是一款健康管理应用程序，那么可以考虑以下几个名称：
1. HealthBoss
2. FitMeUp
3. HealthMate
4. BodyBest
5. HealthLab
其中，HealthBoss强调应用程序可帮助用户成为健康的老板，FitMeUp强调应用程序可帮助用户达到健康生活的目标，HealthMate强调应用程序是用户健康的伙伴，BodyBest强调应用程序可通过健康管理帮助用户获得最佳身体状态，HealthLab强调应用程序可帮助用户进行健康诊断和实验。
一句简短的介绍可包含以下几个要素：
1. 应用程序的名称；
2. 应用程序的主要功能或特点；
3. 应用程序的受众或目标用户。
例如，以上面的例子为基础，可以写出以下几个简短的介绍：
1. HealthBoss – 健康管理，你的健康老板。
2. FitMeUp – 达成健康目标，轻松自在，适合每个人。
3. HealthMat

In [ ]:
# 保存到 Google Drive
save_dir = '/content/drive/MyDrive/belle_dataset'
os.makedirs(save_dir, exist_ok=True)

# 保存前50000条数据作为样本（避免文件过大）
sample_size = min(5000, len(dataset))
sample_dataset = dataset.select(range(sample_size))

# 转换为 DataFrame 并保存
df = pd.DataFrame(sample_dataset)
save_path = os.path.join(save_dir, 'belle_sample_5000.json')
df.to_json(save_path, orient='records', lines=False, force_ascii=False)

print(f"\n样本数据已保存到: {save_path}")
print(f"保存了 {len(df)} 条数据")


样本数据已保存到: /content/drive/MyDrive/belle_dataset/belle_sample_5000.json
保存了 5000 条数据
